## Outline

#### We've trained DyNeMo across 5 independent runs, and each run will have produced slightly different results (modes). The purpose of this script is to identify the "best" run, defined as the one with the lowest variational free energy

In [1]:
# Import packages. We can safely ignore the tensorflow warnings, because we are not using a GPU
import os
import os.path as op
from osl_dynamics.data import Data
from osl_dynamics.models import load
from osl_dynamics.analysis import spectral
from osl_dynamics.inference import modes
import pickle
import numpy as np
import glob
import yaml

In [2]:
# Open the config file and define main directories
with open('0_config.yaml', 'r') as file:
    config = yaml.safe_load(file)
    
subjects_dir = config['dirs']['recon_dir']
proc_dir = config['dirs']['proc_dir']
atlas_dir = config['dirs']['atlas_dir']
model_dir = config['dirs']['model_dir']
results_dir = config['dirs']['results_dir']
#

In [3]:
# Define a helper function to pull the most recent version of a file
# Note that this sleects file based on the date & time that they were last
# modified, not the actual datetime printed in the filename 
def recent_fname(path):
    
    # Takes a partial file path
    files = glob.glob(path)
    most_recent_file = max(files, key=os.path.getmtime)
    
    return most_recent_file



In [4]:
# Get list of subjects. All runs were performed on the same list of subjects, in the same order
subjects_fname = recent_fname(os.path.join(results_dir, 'subject_order_*.txt'))
subjects = np.loadtxt(subjects_fname, dtype=str).tolist()

In [6]:
# Variational free energy was saved to disk on every training run. Loop through runs, reading in the relevant file
# and printing out results
runs = [1,2,3,4,5]
run_labs = []

# Define an empty list to store values
VARs = []

for run in runs:
    run_lab = f'Run {run}'
    run_labs.append(run_lab)

    # Load history for this model
    this_model_fname = f'{model_dir}_run{run}'
    hist_fname = os.path.join(this_model_fname, 'history.pkl')
        
    with open(hist_fname, 'rb') as file:
        history = pickle.load(file)

    # Get variational free energy for this model
    VAR = history['free_energy']

    print(f'Variational free energy for {run_lab}: {VAR}')

    # Store out
    VARs.append(VAR)

# Print out the best run
best_var = np.min(VARs)
best_run = np.where(VARs == best_var)[0][0]
print(f'The run with the lowest VAR is {run_labs[best_run]}')
    

Variational free energy for Run 1: 124.17705535888672
Variational free energy for Run 2: 124.2207260131836
Variational free energy for Run 3: 124.1766128540039
Variational free energy for Run 4: 124.19359588623047
Variational free energy for Run 5: 124.18966674804688
The run with the lowest VAR is Run 3
